In [1]:
import pandas as pd

# データの読み込み
train = pd.read_csv("train.csv")

df = pd.read_csv("df.csv")

train['緯度'] = df['緯度']
train['経度'] = df['経度']

In [2]:
# 東京都23区のリスト
wards = [
    "千代田区", "中央区", "港区", "新宿区", "文京区", "台東区", "墨田区", "江東区",
    "品川区", "目黒区", "大田区", "世田谷区", "渋谷区", "中野区", "杉並区", "豊島区",
    "北区", "荒川区", "板橋区", "練馬区", "足立区", "葛飾区", "江戸川区"
]

In [3]:
# 正規表現を使って「何区」を抽出する関数
def extract_ward(address):
    # 23区リストを元に住所から該当する区を検索
    for ward in wards:
        if ward in address:
            return ward
    return None  # 該当する区がなければ None を返す

In [4]:
# 住所データを含む列に対して処理を適用
train['所在地'] = train['所在地'].apply(extract_ward)

In [5]:
train.drop(['アクセス', '間取り', '築年数', '面積', '建物構造', '所在階', '方角', 'バス・トイレ', 'キッチン', '放送・通信', '室内設備', '駐車場', '周辺環境', '契約期間'], axis=1, inplace=True)

In [6]:
train.head()

,id,賃料,所在地,緯度,経度
0,1,75000,北区,35.758248,139.729155
1,2,76000,中央区,35.662900,139.779212
2,3,110000,渋谷区,35.675147,139.667317
3,4,150000,杉並区,35.700074,139.651613
4,5,74000,葛飾区,35.764544,139.869137


In [7]:
train.to_csv('train_pre2.csv', index=False)

In [8]:
# 緯度・経度が数値型でない場合、float型に変換
#train['緯度'] = pd.to_numeric(train['緯度'], errors='coerce')
#train['経度'] = pd.to_numeric(train['経度'], errors='coerce')

# 区ごとの緯度・経度の平均を計算
mean_lat_long = train.groupby('所在地')[['緯度', '経度']].mean()

# 欠損値を区ごとの平均値で補完する関数
def fill_missing_lat_long(row):
    if pd.isnull(row['緯度']) or pd.isnull(row['経度']):
        # 区ごとの平均値を取得
        mean_values = mean_lat_long.loc[row['所在地']]
        if pd.isnull(row['緯度']):
            row['緯度'] = mean_values['緯度']
        if pd.isnull(row['経度']):
            row['経度'] = mean_values['経度']
    return row

# 欠損値を補完
train = train.apply(fill_missing_lat_long, axis=1)

train.to_csv('train_pre3.csv', index=False)

In [9]:
train.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 31470 entries, 0 to 31469
Data columns (total 5 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   id      31470 non-null  int64  
 1   賃料      31470 non-null  int64  
 2   所在地     31470 non-null  object 
 3   緯度      31470 non-null  float64
 4   経度      31470 non-null  float64
dtypes: float64(2), int64(2), object(1)
memory usage: 1.2+ MB
